# EDA — Section 1: Data Loading & Missing Value Handling
### Hospital Operations & Revenue Risk Intelligence Platform
---

**Phase 2 | Healthcare Analytics Capstone**

This notebook covers the first stage of EDA:
1. Loading the merged dataset
2. Initial inspection (data types, descriptive statistics)
3. Identifying and handling missing values

**Two columns have missing values in this dataset:**

| Column | Missing Count | Strategy |
|--------|--------------|----------|
| `approved_amount` | 1,318 rows | Impute using business logic per `claim_status` and `insurance_provider` |
| `payment_days` | 790 rows | Drop rows — <5% of data, class balance preserved |

> **Note on imputation safety:** All imputation targets *only* the NULL rows using an
> `isnull()` condition. This ensures existing non-null values are never overwritten.


## 0. Setup

Import all required libraries. Only `pandas` and `numpy` are needed for this
section — visualisation libraries will be imported in later EDA sections.


In [ ]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded successfully.")


---
## 1. Load Data

We load the pre-merged dataset from the `processed` folder.
This file was produced by joining `patients`, `visits`, and `billing` on their
respective keys. Each row represents one visit with all patient and billing
attributes attached.


In [ ]:
DATA_PATH = "/Users/narendrafuloria/Desktop/Capstone_IITM/capstone_healthcare_analytics/data/processed/merged_data.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


---
## 2. Initial Inspection

Before handling any missing values, we inspect the dataset structure —
data types, non-null counts, and basic descriptive statistics.
This gives us the full picture of what we're working with.


### 2.1 Descriptive Statistics

`describe()` summarises all numeric columns.
Key things to look for: min/max sanity (e.g. no negative ages or stay hours),
and whether `approved_amount` and `payment_days` show lower `count` values than
the others — which confirms those columns have missing values.


In [ ]:
df.describe()


### 2.2 Column Data Types and Non-Null Counts

`info()` tells us the data type of every column and how many non-null values
each has. Columns where `Non-Null Count` is less than the total row count (25,000)
have missing values.

Note that the three date columns (`registration_date`, `visit_date`, `billing_date`)
currently show as `object` (string) type — we will convert them to `datetime` next.


In [ ]:
df.info()


### 2.3 Convert Date Columns to datetime

The three date columns are loaded as strings by default. We convert them to
`datetime64` so that we can perform date arithmetic later (e.g. checking whether
`visit_date` falls after `registration_date`, computing billing lag).


In [ ]:
date_columns = ["registration_date", "visit_date", "billing_date"]
df[date_columns] = df[date_columns].apply(pd.to_datetime)

print("Date columns converted. Updated dtypes:")
print(df[date_columns].dtypes)


Confirming the full schema after date conversion:


In [ ]:
df.info()


---
## 3. Missing Value Analysis

We count the number of NULL values in every column and calculate the percentage
relative to total rows. This tells us the *scale* of the problem before we
decide what to do about it.


In [ ]:
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

# Show only columns that actually have missing values
missing_report = missing_report[missing_report["Missing Count"] > 0]

print(f"Total rows in dataset: {len(df):,}")
print(f"Columns with missing values: {len(missing_report)}\n")
print(missing_report.to_string())


**Findings:**

- `approved_amount`: **1,318 rows** (~5.3%) have NULL values
- `payment_days`: **790 rows** (~3.2%) have NULL values
- All other columns are complete — no action needed elsewhere

These two columns are handled separately below because the reason for
missingness is different in each case, and therefore the correct fix is different.


---
## 4. Handling Missing Values in `approved_amount`

### Why are values missing?

Before deciding how to handle the 1,318 NULL rows in `approved_amount`, we
need to understand *why* they are missing. If the missingness is tied to
`claim_status`, it may be a business rule rather than a data error — and
the correct fix is different in each case.

We start by breaking down the 1,318 NULL rows by `claim_status`.


### 4.1 Inspect NULL Rows


In [ ]:
# All rows where approved_amount is NULL
null_approved_rows = df[df["approved_amount"].isnull()]

print(f"Total rows with NULL approved_amount: {len(null_approved_rows):,}")
print(f"\nBreakdown by claim_status:")
print(null_approved_rows["claim_status"].value_counts())
print(f"\nAs percentages:")
print(null_approved_rows["claim_status"].value_counts(normalize=True).mul(100).round(2))


**Interpretation:**

The NULL rows are spread across **all three** claim statuses — not just one.
This tells us the missingness is not a single business rule but three different situations:

| `claim_status` | NULL count | Correct action |
|---|---|---|
| **Rejected** | 200 | Set `approved_amount = 0` — rejected claims are approved for nothing |
| **Paid** | 817 | Set `approved_amount = billed_amount × provider's paid approval rate` |
| **Pending** | 301 | Set `approved_amount = billed_amount × provider's pending approval rate` |

We handle each group separately below.

> **Important:** We work on a copy (`df_1`) throughout to protect the original
> dataframe. All imputation is conditioned on `isnull()` so that we only fill
> the missing rows — **existing non-null values are never overwritten**.


### 4.2 Create Working Copy


In [ ]:
# Always work on a copy during cleaning — never modify the raw loaded dataframe
df_1 = df.copy()

print(f"Working copy created: {df_1.shape[0]:,} rows × {df_1.shape[1]} columns")
print(f"NULLs in approved_amount (before any imputation): {df_1['approved_amount'].isnull().sum():,}")


### 4.3 Step 1 — Impute Rejected Claims → 0

**Business logic:** A rejected claim means the insurance provider has refused to
pay anything. The correct `approved_amount` for a rejected claim is ₹0.

**Critical note on the `.loc` condition:**
We use `& df_1["approved_amount"].isnull()` in the filter to target *only the
NULL Rejected rows*. Without this condition, we would overwrite the `approved_amount`
for ALL Rejected rows — including those that already have a valid (non-null) value.


In [ ]:
# Target: Rejected rows WHERE approved_amount is currently NULL
rejected_null_mask = (
    (df_1["claim_status"] == "Rejected") &
    df_1["approved_amount"].isnull()
)

print(f"Rejected rows with NULL approved_amount: {rejected_null_mask.sum():,}")

# Fill with 0
df_1.loc[rejected_null_mask, "approved_amount"] = 0

# Verify
remaining_nulls = df_1["approved_amount"].isnull().sum()
print(f"NULLs in approved_amount after Rejected imputation: {remaining_nulls:,}")
print("  (Expected: 1318 − 200 = 1118 — Paid and Pending NULLs still remain)")


### 4.4 Step 2 — Impute Pending Claims Using Per-Provider Approval Rate

**Business logic:** Pending claims have not been finalised yet — the insurance provider
has partially approved an amount but the process is ongoing. We estimate the missing
`approved_amount` using the *average approval rate that provider has applied to other
Pending claims that already have a known approved_amount*.

**Why per provider, not a flat rate?**
Different insurance providers approve different fractions of the billed amount.
Using a single flat 70% would make all Pending claims look identical (no variance),
which would hurt the Phase 3 model's ability to distinguish between them.
Using the per-provider rate preserves that variation.

We compute the rates from the Pending rows that already have a non-null
`approved_amount` — these serve as the reference group.


In [ ]:
# Compute per-provider approval rate from existing non-null Pending rows
pending_reference = df_1[
    (df_1["claim_status"] == "Pending") &
    df_1["approved_amount"].notna()
].copy()

# Approval rate = approved_amount / billed_amount
pending_reference["approval_rate"] = (
    pending_reference["approved_amount"] / pending_reference["billed_amount"]
)

provider_pending_rates = (
    pending_reference
    .groupby("insurance_provider")["approval_rate"]
    .mean()
    .round(4)
)

print("Average approval rate per insurance provider (Pending claims):")
print(provider_pending_rates.to_string())
print(f"\nReference rows used: {len(pending_reference):,} non-null Pending rows")


In [ ]:
# Impute NULL Pending rows per provider
# Only rows where: claim_status == Pending AND approved_amount is NULL
imputed_count = 0

for provider, rate in provider_pending_rates.items():
    mask = (
        (df_1["claim_status"] == "Pending") &
        df_1["approved_amount"].isnull() &
        (df_1["insurance_provider"] == provider)
    )
    df_1.loc[mask, "approved_amount"] = df_1.loc[mask, "billed_amount"] * rate
    imputed_count += mask.sum()
    print(f"  {provider:<15} | rate = {rate:.4f} | rows imputed = {mask.sum()}")

print(f"\nTotal Pending rows imputed: {imputed_count:,}")
print(f"NULLs in approved_amount after Pending imputation: {df_1['approved_amount'].isnull().sum():,}")
print("  (Expected: 1118 − 301 = 817 — only Paid NULLs remain)")


### 4.5 Step 3 — Impute Paid Claims Using Per-Provider Approval Rate

**Business logic:** A Paid claim means the insurance provider has approved and
disbursed the payment. We check what fraction of the billed amount Paid claims
typically receive, broken down by provider.

We expect this to be close to 100% — if a claim is marked Paid, the provider
generally approves the full billed amount.


In [ ]:
# Compute per-provider approval rate from existing non-null Paid rows
paid_reference = df_1[
    (df_1["claim_status"] == "Paid") &
    df_1["approved_amount"].notna()
].copy()

paid_reference["approval_rate"] = (
    paid_reference["approved_amount"] / paid_reference["billed_amount"]
)

provider_paid_rates = (
    paid_reference
    .groupby("insurance_provider")["approval_rate"]
    .mean()
    .round(4)
)

print("Average approval rate per insurance provider (Paid claims):")
print(provider_paid_rates.to_string())
print(f"\nReference rows used: {len(paid_reference):,} non-null Paid rows")


As expected, all providers show an approval rate of 1.0 (100%) for Paid claims —
so we impute the NULL Paid rows with `billed_amount` directly.
This is equivalent to using a per-provider rate, since all rates are 1.0.


In [ ]:
# Impute NULL Paid rows: approved_amount = billed_amount (100% approval)
paid_null_mask = (
    (df_1["claim_status"] == "Paid") &
    df_1["approved_amount"].isnull()
)

print(f"Paid rows with NULL approved_amount: {paid_null_mask.sum():,}")

df_1.loc[paid_null_mask, "approved_amount"] = df_1.loc[paid_null_mask, "billed_amount"]

print(f"NULLs in approved_amount after Paid imputation: {df_1['approved_amount'].isnull().sum():,}")
print("  (Expected: 0 — all three claim status groups have been handled)")


### 4.6 Verify — approved_amount Fully Imputed


In [ ]:
# Final check: no NULLs should remain in approved_amount
assert df_1["approved_amount"].isnull().sum() == 0, "❌ NULLs still present in approved_amount!"
print("✅ approved_amount: 0 NULL values remaining.")

# Sanity check: approved_amount should never exceed billed_amount
invalid_approvals = df_1[df_1["approved_amount"] > df_1["billed_amount"]]
print(f"✅ Rows where approved_amount > billed_amount: {len(invalid_approvals):,}  (should be 0)")


### 4.7 Create `approval_ratio` Feature

Now that `approved_amount` is fully populated, we create the `approval_ratio`
feature: the fraction of the billed amount that was actually approved.

- **1.0** = fully approved (no revenue leakage)
- **< 1.0** = partial approval (some revenue lost)
- **0.0** = rejected (all revenue lost)

This feature is computed here because it depends on both `approved_amount` and
`billed_amount` being complete. It will be used as an input feature in Phase 3 modeling.

> **Note:** This column is named `approval_ratio` rather than `percentage_approved`
> to be more precise — it is a ratio (0 to 1), not a percentage (0 to 100).


In [ ]:
df_1["approval_ratio"] = df_1["approved_amount"] / df_1["billed_amount"]

print("approval_ratio — Summary Statistics:")
print(df_1["approval_ratio"].describe().round(4))

print(f"\nDistribution by claim_status (mean approval_ratio):")
print(df_1.groupby("claim_status")["approval_ratio"].mean().round(4))


---
## 5. Handling Missing Values in `payment_days`

After handling `approved_amount`, we check the remaining NULL counts.


In [ ]:
print("NULL counts after approved_amount imputation:")
print(df_1.isnull().sum()[df_1.isnull().sum() > 0])


Only `payment_days` still has missing values: **790 rows** (~3.2% of the dataset).

### Strategy: Drop rows

Before dropping, we must verify two things:
1. The percentage is below a safe threshold (commonly <5%)
2. Dropping these rows won't introduce **class imbalance** — i.e. the missing rows
   should have the same `claim_status` distribution as the full dataset.

If missing rows are skewed toward one class (e.g. 90% Rejected), dropping them
would reduce Rejected representation in the training data and bias the model.


### 5.1 Inspect the NULL Rows


In [ ]:
null_payment_rows = df_1[df_1["payment_days"].isnull()]

print(f"Rows with NULL payment_days : {len(null_payment_rows):,}")
print(f"As % of total dataset       : {len(null_payment_rows)/len(df_1)*100:.2f}%")
print(f"\nShape of NULL rows: {null_payment_rows.shape}")


### 5.2 Class Balance Check

We compare the `claim_status` distribution in:
- The **full dataset** — this is the baseline
- The **rows with NULL payment_days** — this should match the baseline

If the proportions are similar, removing these rows is safe. If they differ
significantly (e.g. Rejected is 40% in the NULL group vs 15% overall), dropping
them would distort the class balance in the cleaned dataset.


In [ ]:
# Distribution in the full dataset
full_dist = df_1["claim_status"].value_counts(normalize=True).mul(100).round(3)

# Distribution in the NULL payment_days rows
null_dist = null_payment_rows["claim_status"].value_counts(normalize=True).mul(100).round(3)

# Side-by-side comparison
comparison = pd.DataFrame({
    "Full Dataset (%)": full_dist,
    "NULL Rows (%)": null_dist,
    "Difference (pp)": (null_dist - full_dist).round(3)
})

print("Claim Status Distribution — Full Dataset vs NULL payment_days Rows:")
print(comparison.to_string())


**Interpretation:**

The proportions in the NULL rows closely match the overall dataset distribution.
The difference is less than ~2 percentage points for all classes — this is
within acceptable tolerance. Dropping these 790 rows will not meaningfully
shift the class balance.

**Decision: drop the 790 rows.**

Two conditions are satisfied:
- **Volume**: 790 rows = ~3.2% of 25,000 — well below the 5% threshold
- **Class balance**: distribution in NULL rows ≈ distribution in full dataset


### 5.3 Drop NULL Rows


In [ ]:
rows_before = len(df_1)

df_1.dropna(inplace=True)

rows_after = len(df_1)
rows_dropped = rows_before - rows_after

print(f"Rows before drop : {rows_before:,}")
print(f"Rows dropped     : {rows_dropped:,}")
print(f"Rows after drop  : {rows_after:,}")
print(f"\nNULLs remaining in any column:")
remaining_nulls = df_1.isnull().sum()
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.sum() > 0 else "  ✅ None — dataset is fully clean.")


---
## 6. Final Verification

A complete inspection of the cleaned dataframe before saving.


In [ ]:
print("=" * 50)
print("  Cleaned Dataset — Final Summary")
print("=" * 50)
print(f"\nShape: {df_1.shape[0]:,} rows × {df_1.shape[1]} columns")
print(f"Rows removed from original 25,000: {25000 - df_1.shape[0]:,}")


In [ ]:
df_1.info()


In [ ]:
# Confirm zero nulls across all columns
total_nulls = df_1.isnull().sum().sum()
print(f"Total NULL values across all columns: {total_nulls}")

if total_nulls == 0:
    print("✅ Dataset is fully clean — no missing values.")
else:
    print(f"⚠️  {total_nulls} NULLs remain — investigate before proceeding.")


### Summary of Changes Made

| Column | Original NULLs | Action | Result |
|---|---|---|---|
| `approved_amount` | 1,318 | Imputed: Rejected → 0, Paid → billed_amount, Pending → per-provider rate | 0 NULLs |
| `payment_days` | 790 | Rows dropped (3.2% of data, class balance preserved) | 0 NULLs |
| `approval_ratio` | — | New feature created: `approved_amount / billed_amount` | — |

**Final dataset:** 24,210 rows × 21 columns, zero missing values.


---
## 7. Save Cleaned Dataset

We save the cleaned dataframe to the `interim` folder. This file will be used
as the starting point for all remaining EDA sections and Phase 3 modeling.

The `interim` folder is the right place for this — it is downstream of `processed`
(raw merged data) but not yet the final feature-engineered dataset that will go
into `final` before modeling.


In [ ]:
OUTPUT_PATH = "/Users/narendrafuloria/Desktop/Capstone_IITM/capstone_healthcare_analytics/data/interim/dataset_no_null_values.csv"

df_1.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Cleaned dataset saved to:")
print(f"   {OUTPUT_PATH}")
print(f"\nFile contains: {df_1.shape[0]:,} rows × {df_1.shape[1]} columns")
